# Logistics Data Cleaning and Quality Validation

## tl;dr

- The raw file contains **88,580 rows** but only **86,000 unique shipments**; 2,580 duplicate shipment rows are removed with an auditable record-selection rule.
- Deterministic repairs recover missing weights and delivered dates, normalize transport modes, reconcile delay targets, and correct sign/8x cost corruption without median imputation or generic outlier clipping.
- The cleaned dataset has **86,000 rows, 47 columns, and 25/25 passing validation checks**. Remaining nulls are structurally expected for active or mode-specific records.

## Context & Methods

The source metadata is `logistics data.pdf`; the raw data is `data/logistics.csv`, and `data/orders.csv` is used to validate the order foreign key. The raw CSV is preserved.

### Key Assumptions

- The PDF describes an older selected 32-column schema, while the supplied CSV has 47 columns. Equivalent date, mode, and checkpoint fields are mapped explicitly.
- The supplied five-digit `route_id` format is retained because it is consistent in every row, despite the PDF showing six digits.
- Large values are retained when plausible for the shipment mode/type. Only corruption recoverable from redundant fields is corrected.
- Missing outcome fields remain null for active Delayed, In-Transit, and Failed Delivery records.

## Data

### 1. Run the reproducible cleaning pipeline

In [1]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd().resolve()
while not (repo_root / 'data' / 'logistics.csv').exists():
    if repo_root.parent == repo_root:
        raise FileNotFoundError('Could not locate the project root')
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.clean_logistics import clean_logistics

summary = clean_logistics(
    repo_root / 'data' / 'logistics.csv',
    repo_root / 'data' / 'orders.csv',
    repo_root / 'data' / 'cleaned' / 'logistics_clean.csv',
    repo_root / 'Reports' / 'logistics_data_quality',
)
summary['clean_profile']

{'rows': 86000,
 'columns': 47,
 'unique_shipments': 86000,
 'dispatch_date_min': '2021-01-01T00:00:00',
 'dispatch_date_max': '2024-12-31T00:00:00',
 'validation_checks_passed': 25,
 'validation_checks_total': 25}

### 2. Compare raw and cleaned grain

In [2]:
pd.DataFrame([
    {
        'stage': 'Raw',
        'rows': summary['raw_profile']['rows'],
        'columns': summary['raw_profile']['columns'],
        'unique_shipments': summary['raw_profile']['unique_shipments'],
        'duplicate_shipment_rows': summary['raw_profile']['duplicate_shipment_id_rows_to_remove'],
    },
    {
        'stage': 'Cleaned',
        'rows': summary['clean_profile']['rows'],
        'columns': summary['clean_profile']['columns'],
        'unique_shipments': summary['clean_profile']['unique_shipments'],
        'duplicate_shipment_rows': 0,
    },
])

,stage,rows,columns,unique_shipments,duplicate_shipment_rows
0,Raw,88580,47,86000,2580
1,Cleaned,86000,47,86000,0


## Results

### 3. Review every repair

In [3]:
repair_summary = pd.read_csv(repo_root / 'Reports' / 'logistics_data_quality' / 'repair_summary.csv')
repair_summary.sort_values('rows_affected', ascending=False).reset_index(drop=True)

,repair,rows_affected
0,delay_days_recalculated,5951
1,late_shipments_assigned_unknown_delay_reason,2629
2,duplicate_rows_removed,2580
3,eightfold_fuel_surcharge_corruptions_corrected,2580
4,delivered_actual_arrival_dates_recovered,2335
5,transport_mode_values_normalized,1671
6,shipment_weights_recovered_from_total_divided_...,1670
7,negative_freight_signs_corrected,826
8,delay_flags_recalculated,0
9,transit_days_actual_recalculated,0


### 4. Confirm all business-rule validations

In [4]:
validation = pd.read_csv(repo_root / 'Reports' / 'logistics_data_quality' / 'validation_results.csv')
display(validation)
assert validation['passed'].all(), validation.loc[~validation['passed']]
print(f"Passed {validation['passed'].sum()} of {len(validation)} checks.")

,check,passed,observed,expected,severity_if_failed
0,row_count_matches_unique_raw_shipments,True,86000,86000,Critical
1,shipment_id_is_unique,True,0,0 duplicate keys,Critical
2,no_exact_duplicate_rows,True,0,0 duplicate rows,Critical
3,shipment_id_format,True,0,0 invalid,High
4,order_id_format,True,0,0 invalid,High
5,order_foreign_key_coverage,True,0,0 orphan rows,Critical
6,carrier_id_format,True,0,0 invalid,High
7,route_id_format_supplied_schema,True,0,0 invalid,High
8,transport_mode_allowed_values,True,0,0 invalid,High
9,required_dates_parse,True,0,0 invalid required dates,High


Passed 25 of 25 checks.


### 5. Interpret remaining nulls rather than filling them blindly

In [5]:
pd.DataFrame([
    {
        'column': column,
        'remaining_nulls': count,
        'null_rate_pct': round(100 * count / summary['clean_profile']['rows'], 2),
    }
    for column, count in summary['remaining_nulls'].items()
]).sort_values('remaining_nulls', ascending=False).reset_index(drop=True)

,column,remaining_nulls,null_rate_pct
0,weather_severity,78553,91.34
1,delay_reason,53143,61.79
2,port_of_loading,47264,54.96
3,port_of_discharge,47264,54.96
4,transit_days_actual,24209,28.15
5,actual_arrival_date,24209,28.15
6,delay_days,24209,28.15
7,delay_flag,24209,28.15


## Takeaways

The cleaned output is safe for shipment-level analysis and delay-model preparation at the documented grain. Primary-key uniqueness, order-key coverage, required dates, completed-shipment outcomes, delay target logic, domestic customs rules, weather consistency, and cost arithmetic are all validated.

Carrier and route referential integrity remain open because no carrier or route master was supplied. Analysts should retain missing outcome fields for active shipments and use explicit modeling logic rather than treating those nulls as data loss.